# AI/ML Task 3 - Model Validation, Overfitting Control & Hyperparameter Tuning

**Project:** California House Price Prediction  
**Internship Task:** Task 3 - Model Validation, Overfitting Control & Hyperparameter Tuning  

This notebook is beginner-friendly and contains the complete project workflow:

1. Load California Housing dataset
2. Preprocess and scale features
3. Train baseline regression models
4. Detect overfitting using train vs test scores
5. Apply 5-fold cross-validation
6. Tune Decision Tree using GridSearchCV
7. Compare models using RMSE and R2 Score
8. Select the final model and save it using joblib


## 1. Import Required Libraries

We import libraries for data handling, visualization, machine learning models, evaluation, cross-validation, and model saving.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load and Prepare Dataset

We use the **California Housing Dataset**. The target column is `HousePrice`, which represents median house value.

In [ ]:
data = fetch_california_housing(as_frame=True)

df = pd.concat([data.data, data.target.rename('HousePrice')], axis=1)

print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

## 3. Basic Dataset Exploration

This helps us understand the number of columns, missing values, and statistical summary.

In [ ]:
print('Dataset Information:')
print(df.info())

print('
Missing Values:')
print(df.isnull().sum())

print('
Statistical Summary:')
df.describe()

## 4. Visualize Target Variable

This graph shows the distribution of house prices.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['HousePrice'], bins=40)
plt.xlabel('House Price')
plt.ylabel('Frequency')
plt.title('Distribution of House Prices')
plt.show()

## 5. Separate Features and Target

- `X` contains input features such as income, rooms, population, and location.
- `y` contains the target value, house price.

In [ ]:
X = df.drop('HousePrice', axis=1)
y = df['HousePrice']

print('Features shape:', X.shape)
print('Target shape:', y.shape)

## 6. Feature Scaling

Feature scaling makes all numeric columns comparable. It is especially important for linear models and Ridge Regression.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature scaling completed!')

## 7. Train-Test Split

We split the dataset into:

- 80% training data
- 20% testing data

The test data is used to check how the model performs on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 8. Evaluation Function

We use the following metrics:

- **MAE:** Average absolute error
- **RMSE:** Root Mean Squared Error; lower value is better
- **R2 Score:** Explains how well the model fits the data; higher value is better


In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    
    return train_mae, test_mae, train_rmse, test_rmse, train_r2, test_r2

## 9. Train Baseline Models

We train three models:

1. Linear Regression
2. Ridge Regression
3. Decision Tree Regressor

The Decision Tree is included because it can overfit if not controlled.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42)
}

baseline_results = []

for name, model in models.items():
    train_mae, test_mae, train_rmse, test_rmse, train_r2, test_r2 = evaluate_model(
        model, X_train, X_test, y_train, y_test
    )
    
    baseline_results.append({
        'Model': name,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train R2': train_r2,
        'Test R2': test_r2,
        'Overfitting Gap': test_rmse - train_rmse
    })

baseline_df = pd.DataFrame(baseline_results)
baseline_df

## 10. Overfitting Detection

A model is likely overfitting when:

- Training RMSE is very low
- Testing RMSE is much higher
- The gap between training and testing performance is large

The untuned Decision Tree often overfits because it can keep splitting until it memorizes training data.

In [ ]:
plt.figure(figsize=(10, 5))
bar_width = 0.35
x = np.arange(len(baseline_df['Model']))

plt.bar(x - bar_width/2, baseline_df['Train RMSE'], width=bar_width, label='Train RMSE')
plt.bar(x + bar_width/2, baseline_df['Test RMSE'], width=bar_width, label='Test RMSE')

plt.xlabel('Model')
plt.ylabel('RMSE')
plt.title('Train vs Test RMSE - Overfitting Detection')
plt.xticks(x, baseline_df['Model'], rotation=25)
plt.legend()
plt.show()

## 11. Cross-Validation

Cross-validation gives a more reliable performance estimate than a single train-test split.

Here we use **5-fold cross-validation**:

- Dataset is divided into 5 parts
- Model trains on 4 parts and validates on 1 part
- This process repeats 5 times
- Final score is the average score


In [ ]:
cv_results = []

for name, model in models.items():
    cv_scores = cross_val_score(
        model,
        X_scaled,
        y,
        scoring='neg_root_mean_squared_error',
        cv=5
    )
    
    cv_results.append({
        'Model': name,
        'CV RMSE Mean': -cv_scores.mean(),
        'CV RMSE Std': cv_scores.std()
    })

cv_df = pd.DataFrame(cv_results)
cv_df

## 12. Hyperparameter Tuning with GridSearchCV

Hyperparameters are model settings selected before training.

For Decision Tree, we tune:

- `max_depth`: maximum depth of the tree
- `min_samples_split`: minimum samples required to split a node
- `min_samples_leaf`: minimum samples required at a leaf node

These parameters help reduce overfitting.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation RMSE:', -grid_search.best_score_)

## 13. Evaluate Tuned Decision Tree

Now we evaluate the optimized Decision Tree using the same train and test data.

In [ ]:
best_tree = grid_search.best_estimator_

train_mae, test_mae, train_rmse, test_rmse, train_r2, test_r2 = evaluate_model(
    best_tree, X_train, X_test, y_train, y_test
)

tuned_result = pd.DataFrame([{
    'Model': 'Tuned Decision Tree',
    'Train MAE': train_mae,
    'Test MAE': test_mae,
    'Train RMSE': train_rmse,
    'Test RMSE': test_rmse,
    'Train R2': train_r2,
    'Test R2': test_r2,
    'Overfitting Gap': test_rmse - train_rmse
}])

tuned_result

## 14. Final Model Comparison

We compare all baseline models with the tuned Decision Tree.

In [ ]:
final_results = pd.concat([baseline_df, tuned_result], ignore_index=True)
final_results.sort_values(by='Test RMSE')

## 15. Visualization of Final Results

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(final_results['Model'], final_results['Test RMSE'])
plt.xlabel('Model')
plt.ylabel('Test RMSE')
plt.title('Final Model Comparison - Test RMSE')
plt.xticks(rotation=25)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(final_results['Model'], final_results['Test R2'])
plt.xlabel('Model')
plt.ylabel('Test R2 Score')
plt.title('Final Model Comparison - Test R2 Score')
plt.xticks(rotation=25)
plt.show()

## 16. Final Model Selection

Select the model with:

- Lowest Test RMSE
- Highest Test R2 Score
- Small overfitting gap
- Stable cross-validation performance

In most cases, the tuned Decision Tree performs better than the untuned Decision Tree because tuning controls overfitting.

In [ ]:
best_model_row = final_results.sort_values(by='Test RMSE').iloc[0]
print('Best Model Based on Test RMSE:')
print(best_model_row)

## 17. Save Final Model

We save the final tuned Decision Tree and scaler using `joblib`.

In [ ]:
joblib.dump(best_tree, 'best_house_price_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('Best model saved as best_house_price_model.pkl')
print('Scaler saved as scaler.pkl')

## 18. Final Conclusion

This project demonstrates a professional machine learning workflow for house price prediction. The main focus was not only model accuracy, but also model reliability.

Key conclusions:

- A single train-test split is not enough for reliable model evaluation.
- Cross-validation gives a more stable performance estimate.
- Untuned Decision Tree models can overfit because they may memorize training data.
- GridSearchCV helps find better hyperparameters automatically.
- The final model should be selected using RMSE, R2 Score, overfitting gap, and cross-validation performance.

This makes the model more suitable for real-world use because it generalizes better to unseen data.